# 07. 스트리밍 단계별 출력

> 에이전트의 **단계별 진행 상황**이 보여야 사용자가 기다릴 수 있어요. `values` / `updates` / `messages` 세 가지 스트리밍 모드와 서브그래프 추적까지 다룹니다.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `stream_mode`의 세 가지 모드(`values`, `updates`, `messages`)를 이해하고 차이를 설명할 수 있어요
2. 동기(`graph.stream()`)와 비동기(`graph.astream()`) 스트리밍을 각각 구현할 수 있어요
3. `metadata`를 활용해 특정 노드의 출력만 필터링할 수 있어요
4. `astream_events()`와 태그 기반 필터링으로 세밀한 스트리밍 제어를 할 수 있어요
5. 서브그래프(`subgraphs=True`)의 내부 실행 과정까지 스트리밍으로 추적할 수 있어요

## 사전 지식

- Part 04의 02-Subgraphs: 서브그래프 작성 방법
- Part 02의 04-StateGraph-Basics: StateGraph, 노드, 엣지 구성
- Part 02의 06-Tools-Integration: ToolNode, tools_condition 사용법


## 스트리밍이란?

### 왜 스트리밍이 필요한가요?

ChatGPT를 사용해본 적 있다면, 응답이 한 글자씩 타이핑되듯 나타나는 것을 본 적이 있을 거예요. 이것이 바로 스트리밍이에요. 만약 전체 응답이 완성될 때까지 기다려야 한다면, 긴 답변은 수십 초간 빈 화면만 보게 되죠.

LangGraph의 **스트리밍(Streaming)**은 그래프가 실행되는 동안 중간 결과를 실시간으로 받아볼 수 있는 기능이에요. 이를 통해 사용자 경험이 크게 향상되고, 복잡한 멀티 노드 그래프에서도 진행 상황을 실시간으로 모니터링할 수 있어요.

### 스트리밍 모드 비교

| 모드 | 반환 단위 | 주요 용도 | 비유 |
|------|-----------|----------|------|
| `values` | 노드 실행 후 **전체 State** | 상태 변화 전체 추적, 디버깅 | 매 단계마다 사진을 찍어 앨범에 저장 |
| `updates` | 노드 실행 후 **변경된 부분만** | 노드별 기여분 파악, 효율적 처리 | 매 단계마다 "뭐가 바뀌었는지"만 메모 |
| `messages` | **토큰 단위** LLM 출력 | 실시간 응답 표시, 사용자 UX 향상 | ChatGPT처럼 한 글자씩 타이핑 |

> 🔑 **핵심 개념**: `stream_mode`는 **무엇을** 스트리밍할지 결정해요. `values`는 누적 전체 상태, `updates`는 변경분, `messages`는 LLM 토큰 단위 출력이에요.


## 전체 아키텍처

```mermaid
flowchart LR
    입력["사용자 입력<br>User Input"] --> 그래프["LangGraph<br>그래프"]
    
    그래프 --> values_mode["values 모드<br>전체 State 반환"]
    그래프 --> updates_mode["updates 모드<br>변경분만 반환"]
    그래프 --> messages_mode["messages 모드<br>토큰 단위 스트리밍"]
    
    messages_mode --> 필터링["노드 필터링<br>metadata['langgraph_node']"]
    messages_mode --> 태그필터링["태그 필터링<br>astream_events()"]
    
    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    
    class 입력 input
    class 그래프 process
    class values_mode,updates_mode,messages_mode output
    class 필터링,태그필터링 storage
```

### 동기 vs 비동기 스트리밍

| 방식 | 메서드 | 반복문 | 적합한 환경 |
|------|--------|--------|-------------|
| 동기 | `graph.stream()` | `for chunk in ...` | 스크립트, 간단한 실험 |
| 비동기 | `graph.astream()` | `async for chunk in ...` | 서버, API, 동시 요청 처리 |


## 환경 설정


In [ ]:
# 환경 변수 로드 (.env 파일에서 API 키 등을 읽어요)
from dotenv import load_dotenv
import os

load_dotenv()

# LangSmith 추적 설정 (그래프 실행 과정을 모니터링해요)
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-V1-Tutorial-Part04-07"


## 에이전트 그래프 정의

스트리밍 실습을 위한 도구 검색 에이전트를 만들어볼게요. 이 에이전트는 키워드 뉴스 검색 도구를 사용하고, 결과를 요약해서 답변해요.


In [ ]:
from typing import Annotated, List, Dict
from typing_extensions import TypedDict
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

llm = init_chat_model("openai:gpt-4o-mini")

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 스트리밍 실습용 뉴스 검색 에이전트 정의
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → chatbot → {tools_condition} → tools 또는 END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 1. stream_mode="values" — 전체 상태 스트리밍

`values` 모드는 각 노드가 실행될 때마다 **그래프의 전체 State**를 반환해요.

- 반환 형태: `{state_key: state_value, ...}` (딕셔너리)
- 특징: 누적된 전체 상태를 볼 수 있어서, 이전 노드의 결과도 포함해요
- 적합한 상황: 상태 변화를 전체적으로 추적하고 싶을 때, 디버깅


### 1-1. 동기(Synchronous) 방식 — values


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: stream_mode="values" 동기 스트리밍
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 1-2. 비동기(Asynchronous) 방식 — values


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: stream_mode="values" 비동기 스트리밍
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 1-3. 최종 결과만 추출하기

스트리밍하면서도 최종 결과만 필요한 경우가 있어요. 마지막 청크를 `final_result`에 저장하는 패턴을 사용해요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 최종 결과만 추출하는 패턴
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 2. stream_mode="updates" — 변경분만 스트리밍

`updates` 모드는 각 노드 실행 후 **변경된 부분만** 반환해요.

- 반환 형태: `{node_name: {state_key: updated_value, ...}, ...}`
- 특징: **어떤 노드가** 무엇을 변경했는지 명확히 파악 가능
- 적합한 상황: 노드별 기여분 분석, 부분적인 상태 업데이트 처리

> 🔑 **핵심 개념**: `updates` 모드의 `chunk`는 `{노드이름: {업데이트내용}}` 형태예요. 전체 State가 아니라 해당 노드가 **반환한 값**만 포함해요. 데이터 전송량이 `values` 모드보다 적어요.


### 2-1. 동기(Synchronous) 방식 — updates


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: stream_mode="updates" 동기 스트리밍
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 2-2. 비동기(Asynchronous) 방식 — updates


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: stream_mode="updates" 비동기 스트리밍
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 3. stream_mode="messages" -- 토큰 단위 스트리밍

`messages` 모드는 LLM이 **토큰을 생성할 때마다** 실시간으로 받아볼 수 있어요. ChatGPT에서 글자가 하나씩 나타나는 효과를 구현하는 핵심 모드예요.

- 반환 형태: `(chunk_msg, metadata)` 튜플
  - `chunk_msg`: 실시간 토큰 청크 메시지
  - `metadata`: 노드 정보 딕셔너리 (`langgraph_node`, `langgraph_step`, `langgraph_triggers` 등)
- 적합한 상황: 사용자 인터페이스에서 실시간 응답 표시, 챗봇 UX

### values vs updates vs messages 출력 비교

```
# values 모드 - 전체 State 반환
{"messages": [Human("질문"), AI("답변 전체...")]}

# updates 모드 - 변경분만 반환
{"chatbot": {"messages": [AI("답변 전체...")]}}

# messages 모드 - 토큰 하나씩 반환
("답", metadata)
("변", metadata)
("의", metadata)
(" ", metadata)
("시", metadata)
("작", metadata)
...
```


### 3-1. 동기(Synchronous) 방식 — messages


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: stream_mode="messages" 동기 스트리밍
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 3-2. metadata 구조 확인

`metadata` 딕셔너리에는 어떤 정보가 담겨 있는지 확인해볼게요.

> 🔑 **핵심 개념**: `metadata`의 주요 키들
> - `langgraph_node`: 현재 메시지를 생성한 **노드 이름**
> - `langgraph_step`: 그래프 실행의 **단계 번호** (0부터 시작)
> - `langgraph_triggers`: 현재 노드를 **실행시킨 트리거** (이전 노드 이름)
> - `tags`: LLM에 설정된 **태그 목록**


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: metadata 구조 분석
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 3-3. 비동기(Asynchronous) 방식 — messages


In [ ]:
from langchain.messages import HumanMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: stream_mode="messages" 비동기 스트리밍
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 4. 태그 기반 필터링 스트리밍

### 왜 태그가 필요한가요?

복잡한 그래프에서는 여러 LLM이 각각 다른 역할을 해요. 예를 들어 하나는 뉴스를 검색하고, 다른 하나는 SNS 포스트를 만들어요. 사용자에게는 최종 결과만 보여주고 싶은데, 스트리밍하면 모든 LLM의 토큰이 섞여서 나와요. **태그(tag)**를 붙이면 원하는 LLM 출력만 골라서 스트리밍할 수 있어요.

태그를 설정하는 방법:
```python
llm_with_tools = llm.bind_tools(tools).with_config(tags=["MY_STREAM_TAG"])
```

`astream_events()`를 사용하면 이벤트 기반으로 더 세밀하게 제어할 수 있어요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 태그가 설정된 그래프 재정의
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: astream_events()를 활용한 태그 기반 필터링
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 4-1. 도구 호출 스트리밍 (AIMessageChunk 누적)

도구 호출이 포함된 경우, `AIMessageChunk`를 누적해서 완전한 도구 호출 정보를 재구성할 수 있어요.

> 🔑 **핵심 개념**: LLM이 도구를 호출할 때도 토큰 단위로 스트리밍이 돼요. `tool_call_chunks`가 있으면 도구 호출 중임을 알 수 있어요. `AIMessageChunk` 객체는 `+` 연산자로 누적할 수 있어요.


In [ ]:
from langchain.messages import AIMessageChunk, HumanMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: AIMessageChunk 누적으로 도구 호출 정보 재구성
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. 서브그래프 스트리밍

서브그래프(Subgraph)를 포함한 그래프에서도 스트리밍이 가능해요. `subgraphs=True` 옵션을 사용하면 서브그래프 내부의 실행 과정까지 추적할 수 있어요.

- `subgraphs=False` (기본값): 서브그래프를 단일 노드처럼 처리, 최종 출력만 반환
- `subgraphs=True`: 서브그래프 내부 노드의 업데이트도 포함, `(namespace, chunk)` 튜플 반환

```mermaid
flowchart LR
    사용자["사용자 입력"] --> 메인그래프
    
    subgraph 메인그래프["메인 그래프"]
        뉴스서브["뉴스 검색<br>서브그래프"] --> SNS포스트["SNS 포스트<br>생성 노드"]
    end
    
    subgraph 뉴스서브
        chatbot2["chatbot"] --> tools2["tools"]
        tools2 --> chatbot2
    end
    
    SNS포스트 --> 출력["최종 출력"]
    
    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    class 사용자 input
    class 뉴스서브,SNS포스트,chatbot2,tools2 process
    class 출력 output
```


In [ ]:
news_llm = init_chat_model("openai:gpt-4o-mini").bind_tools(tools).with_config(
sns_llm = init_chat_model("openai:gpt-4o-mini").with_config(

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 서브그래프가 포함된 메인 그래프 정의
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 5-1. 서브그래프 내부 생략 (기본 동작)

기본적으로 서브그래프는 하나의 노드처럼 동작해요. 서브그래프 이름이 노드 이름으로 표시되고, 최종 출력만 반환돼요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 서브그래프 내부 생략 (subgraphs=False가 기본값)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 5-2. 서브그래프 내부 포함 (subgraphs=True)

`subgraphs=True`를 설정하면 서브그래프 내부의 모든 노드 업데이트가 포함돼요.

반환 형태: `(namespace, chunk)` 튜플
- `namespace`: 현재 업데이트가 어느 그래프/서브그래프에서 발생했는지 나타내는 경로 튜플
- `chunk`: 해당 노드의 업데이트

> 🔑 **핵심 개념**: `namespace` 튜플을 파싱하면 업데이트의 출처를 구분할 수 있어요. 빈 튜플 `()`은 메인 그래프, 원소가 있으면 서브그래프예요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 서브그래프 내부 포함 (subgraphs=True)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 5-3. astream_events()로 서브그래프 내부 토큰 스트리밍

`astream_events()`를 `subgraphs=True`와 함께 사용하면, 서브그래프 내부의 LLM 토큰까지 실시간으로 받을 수 있어요.

| 이벤트 | 설명 |
|---|---|
| `on_chat_model_start` | LLM 호출 시작 |
| `on_chat_model_stream` | LLM 토큰 스트리밍 |
| `on_tool_start` | 도구 호출 시작 |
| `on_tool_end` | 도구 호출 완료 |


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: astream_events()로 서브그래프 내부 토큰 스트리밍
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 5-4. 특정 태그만 스트리밍

서브그래프 안에 여러 LLM이 있을 때, 원하는 LLM 출력만 선택적으로 스트리밍할 수 있어요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 특정 태그만 선택해서 스트리밍
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 직접 해보기

아래 TODO 블록을 완성해서 스트리밍 실습을 해보세요!


In [ ]:
# ============================================================
# TODO: 아래 스트리밍 코드를 완성해보세요
#
# 목표: "반도체 관련 최신 뉴스를 검색해줘" 질문에 대해
#       updates 모드로 스트리밍하면서
#       각 노드 이름과 마지막 메시지를 출력해보세요
#
# 힌트:
#   1. inputs에 "반도체" 관련 질문을 설정해요
#   2. graph.stream() 또는 graph.astream()을 사용해요
#   3. stream_mode="updates"로 설정해요
#   4. chunk에서 node_name과 node_value를 추출해요
#
# 예상 결과:
#   [노드: chatbot] - 도구 호출 메시지
#   [노드: tools]   - 뉴스 검색 결과
#   [노드: chatbot] - 최종 요약 답변
# ============================================================


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **`stream_mode="values"`**: 각 노드 실행 후 그래프의 전체 State를 반환해요. `chunk["messages"][-1]`로 새로 추가된 마지막 메시지를 확인할 수 있어요.
- **`stream_mode="updates"`**: 각 노드에서 변경된 부분만 `{노드이름: {변경값}}` 형태로 반환해요. 어떤 노드가 무엇을 했는지 명확히 파악할 수 있어요.
- **`stream_mode="messages"`**: LLM 토큰을 실시간으로 받아요. `(chunk_msg, metadata)` 튜플로 반환되고, `metadata["langgraph_node"]`로 출처 노드를 필터링할 수 있어요.
- **동기 vs 비동기**: `graph.stream()` + `for`, `graph.astream()` + `async for`. 서버 환경에서는 비동기가 성능상 유리해요.
- **태그 기반 필터링**: `.with_config(tags=[...])` + `astream_events()`로 특정 LLM 출력만 선택적으로 스트리밍해요.
- **서브그래프 스트리밍**: `subgraphs=True`로 서브그래프 내부 노드까지 추적해요. `namespace` 튜플로 출처를 구분해요.
- **최종 결과만 추출**: `final_result = None`을 선언하고 스트리밍 루프에서 계속 덮어쓰면, 마지막 청크가 최종 결과가 돼요.


## 다음 노트북 예고

다음 `Part 05의 01-Create-Agent.ipynb`에서는 **`create_agent`를 활용한 에이전트 개발**을 배워요.

지금까지 StateGraph를 직접 구성해서 에이전트를 만들었다면, 이제 LangChain V1의 고수준 API인 `create_agent`를 사용해 더 간결하고 강력한 에이전트를 빠르게 만드는 방법을 배울 거예요. 이 노트북에서 배운 스트리밍 기법은 거기서도 그대로 활용돼요!
